In [1]:
import sys
import warnings

sys.path.append("..")
sys.dont_write_bytecode = True
warnings.filterwarnings("ignore")

In [2]:
import polars as pl
from utils.utils import split_train_test, get_parent_run_id_from_experiment ,compute_generalisation_error_from_run_id_and_experiment_id
from utils.statics import lightgbm_model_name, FEATUERE_ENGINEERING_EXPERIMENT_ID
from utils.preprocessing_handler import PreprocessingHandler
from feature_engineering.RowWiseTransformations import RowWiseTransformations
from feature_engineering.OpenFETransformations import OpenFETransformations
from model.tuning import ModelCVEvaluator
model_type = lightgbm_model_name

In [3]:
passes_df = pl.read_parquet("../data/02-analysis/passes.parquet")

In [4]:
train_df, test_df = split_train_test(passes_df=passes_df)

In [5]:
numerical_columns = [
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "length",
    "angle",
    "duration"
]

columns = train_df.columns
target_column = "outcome"
categorical_columns = [
    col for col in columns if col not in numerical_columns and col != target_column
]

preprocessing_handler = PreprocessingHandler(
    df=train_df, categorical_columns=categorical_columns
)
train_df_temp = preprocessing_handler.preprocess_categorical_columns()
train_df_temp = preprocessing_handler.preprocess_outcome_column()
y_train = train_df_temp.select("outcome").to_series()
X_train = preprocessing_handler.preprocess_outcome_column().drop(pl.col("outcome"))

In [6]:
row_wise_transformations = RowWiseTransformations()
open_fe_transformations = OpenFETransformations(n_features=5)

In [7]:
categorical_columns = ["body_part", "height"]
ohe_columns = ["body_part", "height"]

In [8]:
run_name = f"Feature_engineering_13_ohe"
experiment_name = "Feature Engineering"
model_cv_evaluator = ModelCVEvaluator(model_type=model_type, row_wise_transformations=row_wise_transformations, open_fe_transformations=open_fe_transformations, n_inner_splits=3, n_outer_splits=3, n_trials=50, use_feature_selection=False, use_feature_engineering=True, use_ofe=False, log_feature_importance=True, log_parameter_importance=True, run_name=run_name, experiment_name=experiment_name, ohe_columns=ohe_columns)

In [9]:
result_cv = model_cv_evaluator.get_generalisation_error(X_train=X_train, y_train=y_train)
# result_cv = None

2026-04-02 15:41:17,748 - INFO - Starting training with model lightgbm with the following configuration:
        - 3 inner splits
        - 3 outer splits
        - 50 trials
2026-04-02 15:41:18,298 - INFO - Starting full hyperparameter tuning...


  0%|          | 0/50 [00:00<?, ?it/s]

2026-04-02 15:43:28,244 - INFO - Fitting final model with best hyperparameters...


2026-04-02 15:43:47,178 - INFO - Starting full hyperparameter tuning...


🏃 View run Outer_fold_1 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/1529a1aa1cd746a1981c151c5a3c76e2
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


...

2026-04-02 15:47:59,436 - INFO - Starting full hyperparameter tuning...


🏃 View run Outer_fold_2 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/a5a8c46661164178b26e83344845c7f6
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


  0%|          | 0/50 [00:00<?, ?it/s]

2026-04-02 15:50:30,464 - INFO - Fitting final model with best hyperparameters...


🏃 View run Outer_fold_3 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/bbf1413fe49049c996819c73c91400c0
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708
🏃 View run Feature_engineering_13_ohe_lightgbm at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/363ae53267e84aff9638b3b18726bec6
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


In [10]:
parent_run_id = get_parent_run_id_from_experiment(result=result_cv, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID)
compute_generalisation_error_from_run_id_and_experiment_id(parent_run_id=parent_run_id, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID)

Number of outer folds: 3
95% confidence interval for best estimate of generalisation: 0.2711627880054668 ± 0.004163082240990599
